In [1]:
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device)

cuda


In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip # MovieLens 데이터셋 다운로드
!unzip ml-latest-small.zip

--2026-05-31 08:17:07--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  3.38MB/s    in 0.3s    

2026-05-31 08:17:08 (3.38 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [3]:
import torch

users = []
items = []
ratings = []

with open("ml-latest-small/ratings.csv", "r") as f:
    print(f.readline()) # 헤더 라인 출력 (데이터 처리 시 건너뛰기 위함)

    for line in f:
        uid, mid, rating, _ = line.strip().split(",")

        users.append(int(uid))
        items.append(int(mid))
        ratings.append(float(rating))

users = torch.tensor(users)
items = torch.tensor(items)
ratings = torch.tensor(ratings)

userId,movieId,rating,timestamp



In [4]:
shuffled_indices = torch.randperm(len(users))

train_size = int(len(users) * 0.8)

train_indices = shuffled_indices[:train_size]
test_indices = shuffled_indices[train_size:]

users_train = users[train_indices].to(device)
items_train = items[train_indices].to(device)
ratings_train = ratings[train_indices].to(device)

users_test = users[test_indices].to(device)
items_test = items[test_indices].to(device)
ratings_test = ratings[test_indices].to(device)

# Neural Collaborative Filtering

 NCF 모델 생성

In [30]:
import torch.nn as nn

dropout_rate = 0.3 # 드롭아웃 비율
n_factors = 10 # 잠재 요인 수

n_users = int(max(users)) + 1 # 총 사용자 수
n_items = int(max(items)) + 1 # 총 아이템 수

mean = ratings_train.mean() # 훈련 평점 평균

P = torch.randn(
    n_users, n_factors,
    requires_grad=True, device=device
) # 사용자 임베딩 P 초기화

Q = torch.randn(
    n_items, n_factors,
    requires_grad=True, device=device
) # 아이템 임베딩 Q 초기화

model = nn.Sequential( # NCF 모델 정의
    nn.Linear(n_factors * 2, 10),
    nn.ReLU(),
    nn.Dropout(dropout_rate),
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Dropout(dropout_rate),
    nn.Linear(5, 1)
).to(device) # 모델을 장치로 이동

NCF 학습

In [32]:
mse = nn.MSELoss() # 평균 제곱 오차(MSE) 손실 함수 정의

optim = torch.optim.Adam(list(model.parameters())+[P,Q],lr=0.09) # Adam 옵티마이저 설정 (모델 파라미터 및 임베딩 P, Q 학습)

for epoch in range(3001):

    model.train() # 훈련 모드로 설정

    x = torch.cat((P[users_train], Q[items_train]),dim=1) # 사용자 임베딩과 아이템 임베딩을 결합하여 모델 입력 생성

    h = model(x) # 모델을 통해 평점 예측

    cost = mse(h, ratings_train.view(-1, 1)) # 예측 평점과 실제 평점 간의 MSE 손실 계산

    optim.zero_grad() # 이전 기울기 초기화
    cost.backward() # 역전파를 통해 기울기 계산
    optim.step() # 모델 파라미터 업데이트

    with torch.no_grad(): # 평가 단계에서는 기울기 계산 비활성화

        if epoch % 100 == 0:

            model.eval() # 평가 모드로 설정

            x_train = torch.cat((P[users_train], Q[items_train]),dim=1) # 훈련 세트 입력

            h_train = model(x_train) # 훈련 세트 예측
            train_cost = mse(h_train,ratings_train.view(-1, 1)) # 훈련 MSE 계산

            x_test = torch.cat((P[users_test], Q[items_test]),dim=1) # 테스트 세트 입력

            h_test = model(x_test) # 테스트 세트 예측

            test_cost = mse( # 테스트 MSE 계산
                h_test,
                ratings_test.view(-1, 1)
            )

            print( # 에포크, 훈련 MSE, 테스트 MSE 출력
                f"epoch: {epoch}, "
                f"train_mse: {train_cost.item()}, "
                f"test_mse: {test_cost.item()}"
            )

epoch: 0, train_mse: 1.3627227544784546, test_mse: 1.381584644317627
epoch: 100, train_mse: 0.5813053250312805, test_mse: 0.8440905809402466
epoch: 200, train_mse: 0.5832958221435547, test_mse: 0.8459857106208801
epoch: 300, train_mse: 0.5811110138893127, test_mse: 0.8462940454483032
epoch: 400, train_mse: 0.5750137567520142, test_mse: 0.845564603805542
epoch: 500, train_mse: 0.5908823609352112, test_mse: 0.8488258719444275
epoch: 600, train_mse: 0.5815515518188477, test_mse: 0.8455653786659241
epoch: 700, train_mse: 0.5940426588058472, test_mse: 0.847423255443573
epoch: 800, train_mse: 0.5744867324829102, test_mse: 0.846078634262085
epoch: 900, train_mse: 0.5844638347625732, test_mse: 0.844977080821991
epoch: 1000, train_mse: 0.5750219225883484, test_mse: 0.8450298309326172
epoch: 1100, train_mse: 0.5707223415374756, test_mse: 0.84599769115448
epoch: 1200, train_mse: 0.6124444007873535, test_mse: 0.8550139665603638
epoch: 1300, train_mse: 0.5639872550964355, test_mse: 0.84824264049530

# AutoRec

AutoRec용 Rating Matrix 만들기

In [8]:
rating_matrix_train = torch.zeros(n_users, n_items) # 훈련용 평점 행렬 초기화

for uid, mid, rating in zip(users_train,items_train,ratings_train): # 훈련 데이터의 사용자, 아이템, 평점 반복
    rating_matrix_train[uid, mid] = rating # 해당 위치에 평점 기록
rating_matrix_train = rating_matrix_train.to(device) # 훈련 평점 행렬을 장치로 이동

rating_matrix_test = torch.zeros(n_users, n_items) # 테스트용 평점 행렬 초기화

for uid, mid, rating in zip(users_test,items_test,ratings_test): # 테스트 데이터의 사용자, 아이템, 평점 반복
    rating_matrix_test[uid, mid] = rating # 해당 위치에 평점 기록

rating_matrix_test = rating_matrix_test.to(device) # 테스트 평점 행렬을 장치로 이동

AutoRec 모델 생성

In [9]:
import torch.nn as nn

n_factors = 500 # AutoRec 모델의 은닉층 크기 (잠재 요인 수)

model = nn.Sequential( # AutoRec 모델 정의
    nn.Linear(n_users, n_factors), # 입력층 (사용자 수 -> 잠재 요인 수)
    nn.Sigmoid(),
    nn.Linear(n_factors, n_users) # 출력층 (잠재 요인 수 -> 사용자 수)
).to(device) # 모델을 지정된 장치로 이동

AutoRec 학습

In [10]:
optim = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=0.001 # 가중치 감소 (L2 정규화)
)

mse = nn.MSELoss() # 평균 제곱 오차(MSE) 손실 함수 정의

non_zeros_train = rating_matrix_train != 0 # 훈련 평점 행렬에서 0이 아닌 값(실제 평점)의 위치 마스크
non_zeros_test = rating_matrix_test != 0 # 테스트 평점 행렬에서 0이 아닌 값의 위치 마스크

for epoch in range(1001):

    h = model(rating_matrix_train.T) # 훈련 평점 행렬을 모델에 입력하여 예측값 계산 (AutoRec은 사용자 행렬의 전치를 입력으로 받음)

    cost = mse( # 손실(cost) 계산
        h.T[non_zeros_train], # 예측값 중 0이 아닌 부분
        rating_matrix_train[non_zeros_train] # 실제 훈련 평점 중 0이 아닌 부분
    )

    optim.zero_grad() # 옵티마이저의 기울기 초기화
    cost.backward() # 역전파를 통해 기울기 계산
    optim.step() # 모델 파라미터 업데이트

    with torch.no_grad(): # 기울기 계산을 비활성화 (평가 단계)

        if epoch % 100 == 0:

            h_train = model(rating_matrix_train.T) # 훈련 세트에 대한 예측

            train_mse = mse( # 훈련 MSE 계산
                h_train.T[non_zeros_train],
                rating_matrix_train[non_zeros_train]
            )

            h_test = model(rating_matrix_test.T) # 테스트 세트에 대한 예측

            test_mse = mse( # 테스트 MSE 계산
                h_test.T[non_zeros_test],
                rating_matrix_test[non_zeros_test]
            )

            print( # 에포크, 훈련 MSE, 테스트 MSE 출력
                f"epoch: {epoch}, "
                f"train_mse: {train_mse.item()}, "
                f"test_mse: {test_mse.item()}"
            )

epoch: 0, train_mse: 1.8532013893127441, test_mse: 1.9605194330215454
epoch: 10, train_mse: 1.01719331741333, test_mse: 1.2836114168167114
epoch: 20, train_mse: 0.9833441376686096, test_mse: 1.227223515510559
epoch: 30, train_mse: 0.7890758514404297, test_mse: 0.996330976486206
epoch: 40, train_mse: 0.6885154247283936, test_mse: 0.8654630780220032
epoch: 50, train_mse: 0.6418637037277222, test_mse: 0.8398338556289673
epoch: 60, train_mse: 0.611246645450592, test_mse: 0.8391085267066956
epoch: 70, train_mse: 0.5842607021331787, test_mse: 0.8207852840423584
epoch: 80, train_mse: 0.5598897337913513, test_mse: 0.7956869602203369
epoch: 90, train_mse: 0.5348884463310242, test_mse: 0.7886569499969482
epoch: 100, train_mse: 0.5078879594802856, test_mse: 0.7752359509468079
epoch: 110, train_mse: 0.4791075587272644, test_mse: 0.7590488791465759
epoch: 120, train_mse: 0.4499486982822418, test_mse: 0.744283139705658
epoch: 130, train_mse: 0.4218677282333374, test_mse: 0.727252185344696
epoch: 140